# Automated CDC Scenario Testing

This notebook uses the automated `load_and_merge_cdc_to_delta()` function to test CDC scenarios.

## Features
- ✅ Auto-detects primary keys from CockroachDB
- ✅ Auto-detects column families
- ✅ Loads, transforms, merges, and writes in one call
- ✅ Verifies results automatically
- ✅ Compares with source files

## Prerequisites
- Unity Catalog Volume with synced test data
- Configuration files: `.env/cockroachdb_credentials.json` and `.env/cockroachdb_pipelines.json`
- Data synced via `test_cdc_matrix.sh` (auto-syncs to volume subdirectories)


## Step 1: Setup and Configuration


In [ ]:
if "dbutils" not in vars():
    raise RuntimeError("This notebook must be run in Databricks Connect or workspace with dbutils available")
if "spark" not in vars():
    raise RuntimeError("This notebook must be run in Databricks Connect or workspace with Spark available")

In [ ]:
import json
import os
import sys
import importlib

# Add parent directory to path
parent_dir = os.path.abspath("../..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import cockroachdb
importlib.reload(cockroachdb)
from cockroachdb import load_crdb_config, load_and_merge_cdc_to_delta, cleanup_test_checkpoint

print("="*80)
print("CONFIGURATION SETUP")
print("="*80)

# Load configuration files
git_root = os.path.abspath("../../..")
cockroach_dir = f"{git_root}/sources/cockroachdb"
crdb_json_path = f"{cockroach_dir}/.env/cockroachdb_credentials.json"
pipeline_json_path = f"{cockroach_dir}/.env/cockroachdb_pipelines.json"

crdb_config = load_crdb_config(crdb_json_path)

with open(pipeline_json_path, 'r') as f:
    pipeline_config = json.load(f)

print("\n✅ Configuration loaded!")
print("="*80)


## Step 2: Configure Test Scenario

Update these variables to test different scenarios from `test_cdc_matrix.sh`:


In [ ]:
# ============================================================================
# 🔧 CHANGE THIS TO TEST DIFFERENT SCENARIOS
# ============================================================================

# ============================================================================
# Available test scenarios (JSON first, then Parquet):
#   - "test-json_usertable_with_split"        ⭐ Tests merge with column families
#   - "test-json_usertable_no_split"
#   - "test-json_simple_test_with_split"
#   - "test-json_simple_test_no_split"
#   - "test-parquet_usertable_with_split"
#   - "test-parquet_usertable_no_split"
#   - "test-parquet_simple_test_with_split"
#   - "test-parquet_simple_test_no_split"
# ============================================================================


# Test scenario (subdirectory name from test_cdc_matrix.sh)
TEST_FORMAT="json"   # json | parquet
TEST_NAME="usertable_with_split"

# Test version (which test run to analyze)
TEST_VERSION = 0  # 0=latest, 1=second newest, -1=oldest

TEST_SCENARIO = f"test-{TEST_FORMAT}_{TEST_NAME}"  # ⭐ Change this!


In [ ]:
# Parse test scenario to extract components
# Uses centralized parse_test_scenario() from cockroachdb.py
# Pattern: test-{format}_{table_name}_{split_info}
# Examples:
#   "test-parquet_simple_test_no_split" → format='parquet', table_name='simple_test', has_split=False
#   "test-json_usertable_with_split" → format='json', table_name='usertable', has_split=True
from cockroachdb import parse_test_scenario

scenario = parse_test_scenario(TEST_SCENARIO)

# Extract table name from scenario
SOURCE_TABLE = scenario.table_name

# CockroachDB connection defaults (used for schema auto-detection)
# These are NOT part of the scenario name - they're configuration
CRDB_CATALOG = "defaultdb"  # CockroachDB database/catalog
CRDB_SCHEMA = "public"      # CockroachDB schema

print(f"✅ Parsed scenario: {scenario.scenario_name}")
print(f"   Format: {scenario.format}")
print(f"   Table: {SOURCE_TABLE}")
print(f"   Split: {scenario.split_info}")
print(f"   Using catalog: {CRDB_CATALOG}")
print(f"   Using schema: {CRDB_SCHEMA}")
print()

# Derived configuration
CATALOG = pipeline_config["catalog"]
SCHEMA = pipeline_config["schema"]
VOLUME_NAME = pipeline_config["volume_name"]

# Volume path prefix (timestamp will be resolved based on TEST_VERSION)
# This allows testing different test runs without changing paths
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/{TEST_FORMAT}/{CRDB_CATALOG}/{CRDB_SCHEMA}/{TEST_SCENARIO}"

# Target Delta table
TARGET_TABLE = f"{SOURCE_TABLE}_{TEST_SCENARIO.replace('-', '_')}_delta"
TARGET_TABLE_PATH = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"

print("="*80)
print("TEST CONFIGURATION")
print("="*80)
print(f"Test scenario: {TEST_SCENARIO}")
print(f"Test version: {TEST_VERSION} (0=latest, -1=oldest)")
print(f"Source table: {SOURCE_TABLE}")
print(f"Volume path prefix: {VOLUME_PATH}")
print(f"  (Timestamp will be resolved automatically)")
print(f"Target table: {TARGET_TABLE_PATH}")
print("="*80)

## Step 3: Run Automated Test

This single function call:
- Auto-detects primary keys and column families
- Loads data with Autoloader
- Applies CDC transformations
- Merges column family fragments
- Writes to Delta table
- Verifies results
- Compares with source files


In [ ]:
# Run automated test
result = load_and_merge_cdc_to_delta(
    source_table=SOURCE_TABLE,
    volume_path=VOLUME_PATH,
    target_table_path=TARGET_TABLE_PATH,
    crdb_config=crdb_config,
    catalog=CRDB_CATALOG,
    schema=CRDB_SCHEMA,
    spark=spark, 
    dbutils=dbutils,
    clear_checkpoint=True,   # Set to False if appending data
    verify=True,             # Verify Delta table
    compare_source=True,     # Compare with source files
    debug=True,              # Show detailed progress
    version=TEST_VERSION     # Which test run to use (0=latest, -1=oldest)
)


# Step 4 (Optional)

In [ ]:
# Quick cleanup after inspecting results
# This is faster than clear_checkpoint=True because:
# - You can inspect the results first
# - Only deletes what's needed
# - Can keep table for further inspection if desired

# Option 1: Clean checkpoint only (keep table for inspection)
cleanup_test_checkpoint(
    volume_path=VOLUME_PATH,
    dbutils=dbutils,
    version=TEST_VERSION,
    drop_table=False  # Keep table
)

# Option 2: Full cleanup (checkpoint + table)
# cleanup_test_checkpoint(
#     volume_path=VOLUME_PATH,
#     target_table_path=TARGET_TABLE_PATH,
#     dbutils=dbutils,
#     spark=spark,
#     version=TEST_VERSION,
#     drop_table=True  # Drop table too
# )

print("\n✅ Ready for next test iteration!")


## Step 4: Review Results


In [ ]:
print("="*80)
print("TEST RESULTS")
print("="*80)
print(f"Success: {result['success']}")
print(f"Primary keys: {result['primary_keys']}")
print(f"Has column families: {result['has_column_families']}")
print(f"Delta table rows: {result['delta_count']:,}")
print(f"Source file rows: {result['source_count']:,}")
print(f"Match: {result['match']} {'✅' if result['match'] else '⚠️'}")
print("="*80)

if result['match']:
    print("\n✅✅✅ TEST PASSED! ✅✅✅")
    print("Column family merge worked correctly!")
else:
    diff = result['delta_count'] - result['source_count']
    print(f"\n⚠️  TEST FAILED: {diff:+,} row difference")
    print("Review the logs above for details.")


## Step 5: Query Delta Table (Optional)


In [ ]:
# Display sample data
display(spark.table(TARGET_TABLE_PATH).limit(10))


In [ ]:
# Display operation breakdown
display(spark.table(TARGET_TABLE_PATH).groupBy("_cdc_operation").count())


# Debug Code

In [ ]:
from pyspark.sql import functions as F

result_df = spark.table("main.robert_lee_cockroachdb.usertable_test_json_usertable_no_split_delta")

print("\n📊 Operation Breakdown:")
result_df.groupBy("_cdc_operation").count().orderBy("_cdc_operation").show()

print(f"\n📈 Total rows: {result_df.count()}")
print(f"Expected: 1300 (1000 SNAPSHOT + 200 INSERT + 100 UPDATE)")

unknown_count = result_df.filter(F.col("_cdc_operation") == "UNKNOWN").count()
print(f"\n❓ UNKNOWN rows: {unknown_count} (should be 0)")

if result_df.count() == 1300 and unknown_count == 0:
    print("\n✅ FIX VALIDATED! All rows correctly classified.")
else:
    print(f"\n⚠️ Row count mismatch. Delta: {result_df.count() - 1300}")

In [ ]:
from pyspark.sql import functions as F

result_df = spark.table("main.robert_lee_cockroachdb.usertable_test_json_usertable_no_split_delta")

# Check for rows that might be misclassified DELETEs
print("🔍 Checking for potential misclassified DELETE rows...")
print("\nLooking for SNAPSHOT rows where after might be empty/null:")

potential_deletes = result_df.filter(
    (F.col("_cdc_operation") == "SNAPSHOT") &
    ((F.col("_after_json").isNull()) | 
     (F.col("_after_json") == "null") | 
     (F.col("_after_json") == "{}"))
)

print(f"\nSNAPSHOT rows with empty after: {potential_deletes.count()}")

if potential_deletes.count() > 0:
    print("\nSample of these rows:")
    potential_deletes.select(
        "ycsb_key",
        "_cdc_operation",
        "_after_json",
        "_before_json",
        "_debug_after_first_10",
        "_debug_before_first_10",
        "updated"
    ).show(10, truncate=False)

## Summary

### What Just Happened

1. **Auto-detection**: Primary keys and column families detected from CockroachDB
2. **Loading**: Parquet files loaded from Unity Catalog Volume with Autoloader
3. **Transformation**: CDC metadata added (operation type, timestamp, source file)
4. **Merging**: Column family fragments merged into complete rows
5. **Writing**: Data written to Delta table with streaming aggregation
6. **Verification**: Row counts compared between Delta table and source files

### Next Steps

- Test more scenarios by changing `TEST_SCENARIO` in Step 2
- Compare results across different format/split combinations
- Validate that `test-parquet_usertable_with_split` produces correct count (not 11x inflation)

### Key Learnings

- **Automation**: One function call replaces 8 manual steps
- **Auto-detection**: No need to manually specify primary keys or check for column families
- **Verification**: Built-in validation ensures data integrity
- **Flexibility**: All steps can be controlled with optional parameters
